In [16]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [17]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

model = init_chat_model("google_genai:gemini-2.5-flash")

model.invoke([HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?")])

AIMessage(content='저는 실시간 주식 데이터를 가지고 있지 않아 현재 시점에서 테슬라 주가가 한 달 전에 비해 올랐는지 내렸는지 정확히 알려드릴 수 없습니다.\n\n가장 정확한 정보를 얻으시려면 다음 방법을 이용해주세요:\n\n1.  **구글 검색:** "테슬라 주가" 또는 "TSLA 주가" 검색 후 주식 정보 사이트(구글 파이낸스, 네이버 금융, 카카오 증권 등)에서 **1개월 차트**를 확인해 보세요.\n2.  **증권사 앱/웹사이트:** 사용하시는 증권사 앱이나 웹사이트에서 테슬라(TSLA) 종목을 검색하여 1개월간의 주가 변동을 확인하실 수 있습니다.\n\n대부분의 주식 정보 사이트에서는 1개월, 3개월, 1년 등 기간별 주가 차트와 변동률을 제공합니다.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--63acfecf-0575-440c-a9f7-5afb255e1428-0', usage_metadata={'input_tokens': 17, 'output_tokens': 530, 'total_tokens': 547, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 335}})

### ⌚시계 도구

In [18]:
from langchain.tools import tool
from datetime import datetime
from zoneinfo import ZoneInfo

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    target_timezone = ZoneInfo(timezone)
    now = datetime.now(target_timezone).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time

#### 파이단틱(Pydantic)

*   **데이터 유효성 검사**: 런타임에 데이터 유효성을 강제하여 오류를 조기에 발견하고 안정적인 애플리케이션을 구축할 수 있게 한다. 
*   **설정 관리**: 환경 변수, 파일 등에서 애플리케이션 설정을 로드하고 유효성을 검사하여 안전하고 구성 가능한 애플리케이션을 만들 수 있게 한다.
*   **데이터 직렬화 및 역직렬화**: Python 객체를 JSON과 같은 형식으로 쉽게 변환(직렬화)하고 다시 Python 객체로 역변환(역직렬화)할 수 있게 한다.
*   **문서화**: Pydantic 모델은 OpenAPI(Swagger)와 같은 도구에서 자동으로 API 문서를 생성하는 데 사용될 수 있게 한다.

In [19]:
# 도구의 스키마(schema) 정의, LLM이 정확한 도구 호출 생성 가능
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: TSLA)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

#### Yahoo Finance API 간단 사용법

In [20]:
import yfinance as yf

# 테슬라 주식 데이터 가져오기
tsla = yf.Ticker("TSLA")

# 1년간의 일별 주가 데이터
hist = tsla.history(period="1y")

In [21]:
type(hist)

pandas.core.frame.DataFrame

In [22]:
hist

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2025-04-03 00:00:00-04:00,265.290009,276.299988,261.510010,267.279999,136174300,0.0,0.0
2025-04-04 00:00:00-04:00,255.380005,261.000000,236.000000,239.429993,181229400,0.0,0.0
2025-04-07 00:00:00-04:00,223.779999,252.000000,214.250000,233.289993,183453800,0.0,0.0
2025-04-08 00:00:00-04:00,245.000000,250.440002,217.800003,221.860001,171603500,0.0,0.0
2025-04-09 00:00:00-04:00,224.690002,274.690002,223.880005,272.200012,219433400,0.0,0.0
...,...,...,...,...,...,...,...
2026-03-27 00:00:00-04:00,369.690002,369.859985,359.470001,361.829987,62065700,0.0,0.0
2026-03-30 00:00:00-04:00,365.859985,367.290009,352.140015,355.279999,67954400,0.0,0.0
2026-03-31 00:00:00-04:00,361.510010,373.329987,361.000000,371.750000,75534900,0.0,0.0


In [23]:
# 테슬라 주식 데이터 가져오기
tsla = yf.Ticker("TSLA")

# 특정 기간 (예: 2024년 1월 1일 ~ 2월 29일)
start_date = "2025-10-01"
end_date = "2025-10-31"
specific_hist = tsla.history(start=start_date, end=end_date)

In [24]:
specific_hist

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2025-10-01 00:00:00-04:00,443.799988,462.290009,440.750000,459.459991,98122300,0.0,0.0
2025-10-02 00:00:00-04:00,470.540009,470.750000,435.570007,436.000000,137009000,0.0,0.0
2025-10-03 00:00:00-04:00,443.290009,446.769989,416.579987,429.829987,133188200,0.0,0.0
2025-10-06 00:00:00-04:00,440.750000,453.549988,436.690002,453.250000,85324900,0.0,0.0
2025-10-07 00:00:00-04:00,447.820007,452.679993,432.450012,433.089996,102296100,0.0,0.0
2025-10-08 00:00:00-04:00,437.570007,441.329987,425.230011,438.690002,71192100,0.0,0.0
2025-10-09 00:00:00-04:00,431.809998,436.350006,426.179993,435.540009,69339900,0.0,0.0
2025-10-10 00:00:00-04:00,436.540009,443.130005,411.450012,413.489990,112107900,0.0,0.0
2025-10-13 00:00:00-04:00,423.529999,436.890015,419.700012,435.899994,79552800,0.0,0.0


In [25]:
# uv add tabulate
print(specific_hist.to_markdown())

| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |
|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|
| 2025-10-01 00:00:00-04:00 | 443.8  | 462.29 | 440.75 |  459.46 | 9.81223e+07 |           0 |              0 |
| 2025-10-02 00:00:00-04:00 | 470.54 | 470.75 | 435.57 |  436    | 1.37009e+08 |           0 |              0 |
| 2025-10-03 00:00:00-04:00 | 443.29 | 446.77 | 416.58 |  429.83 | 1.33188e+08 |           0 |              0 |
| 2025-10-06 00:00:00-04:00 | 440.75 | 453.55 | 436.69 |  453.25 | 8.53249e+07 |           0 |              0 |
| 2025-10-07 00:00:00-04:00 | 447.82 | 452.68 | 432.45 |  433.09 | 1.02296e+08 |           0 |              0 |
| 2025-10-08 00:00:00-04:00 | 437.57 | 441.33 | 425.23 |  438.69 | 7.11921e+07 |           0 |              0 |
| 2025-10-09 00:00:00-04:00 | 431.81 | 436.35 | 426.18 |  435.54 | 6.93399e+07 |           0 |          

### 📈 주식 종목 가격 조회 도구

In [26]:
@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown() 

    return history_md

In [27]:
from dataclasses import dataclass

@dataclass
class UserContext:
    user_id: str

In [28]:
# create_tool_calling_agent: 도구 호출 기능이 있는 에이전트를 생성하는 함수
# AgentExecutor: 생성된 에이전트의 실행을 담당하는 클래스
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_core.prompts import ChatPromptTemplate

# 도구들을 tools 리스트에 추가
tools = [get_current_time, get_yf_stock_history]

# 에이전트 생성
agent = create_agent(
    model,
    tools=tools,
    context_schema=UserContext,
    system_prompt="너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."
)

In [29]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"}]},
    context=UserContext(user_id="user123")
)

In [30]:
result

{'messages': [HumanMessage(content='테슬라는 한달 전에 비해 주가가 올랐나 내렸나?', additional_kwargs={}, response_metadata={}, id='cd3c60e8-eeed-4d30-a75d-c0135af61ed3'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_yf_stock_history', 'arguments': '{"stock_history_input": {"ticker": "TSLA", "period": "1mo"}}'}, '__gemini_function_call_thought_signatures__': {'f13694ce-148f-4ac2-8800-5a1468a6488e': 'CuAEAb4+9vtvBuXXjeI2BSt1frVrN05P0wKq30WrEBLRisazUL/kXGzbxMp9mlGK0LLryHv6SbIFJAmeF535Sg12AOve1ub8ObQsobrlZQodDcfSeOICqgXe9MCsR+wuBDblM42aMVvi2nyDVLfDJh/6NmhlAAykQE3KyPwOsP+5ah7nx95a5eTHgqeyZKrDwaSakxq8nu+HM5m11CUSuo2UDyy+K/AMVVWXJd2wlQ9zfNmhr/xKsDJE61hSDifkzxMAuLvfOCE+31PA5AtDabuCYCnt4zXukCc/gKCc5lg2RIrlO7kUhoLDzAxjFgdJs7d2UkxY5jDAyznq1qMBS7ye6Uh8i2aGnm0Bulxz3yV5SMzm4ycjS1rReJlZvpcdazucS2PlgSkzqUBHH66G3E9t87UoT4rTWMCqTyicu2vStfYc+3avkyvXdDzQjVx/rjnPboniJsJOJAecauTK1UpGSy3u2OIne5l2qyF4DNITL1voDbX3Extn/+eWEmGShJesVSwpTAAVfif1m+qPrtPDmPxesUcTXrojv+MMAAFuraWmnKPk+bVRdYYDCqKMPE3Et+hDsj1